# 01 — Exploratory Data Analysis
**Telco Customer Churn**

This notebook performs comprehensive EDA on the Telco Customer Churn dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'

## Load Data

In [ ]:
df = pd.read_csv('../data/telco-customer-churn.csv')
print('Shape:', df.shape)
assert df.shape == (7043, 21), f'Expected (7043, 21), got {df.shape}'

In [ ]:
df.info()

In [ ]:
df.describe()

## Missing Values & Duplicates

In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\n⚠️ Note: df.isna() shows 0 for TotalCharges because blanks are " " (space), not NaN!')
print(f'Blank TotalCharges count: {(df["TotalCharges"] == " ").sum()}')

## Churn Distribution

In [ ]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))

### Chart 1: Churn Count

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1FA97A', '#E5484D']
df['Churn'].value_counts().plot(kind='bar', color=colors, edgecolor='white', ax=ax)
ax.set_title('Churn Distribution', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_xlabel('Churn', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_xticklabels(['No', 'Yes'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: The dataset is imbalanced — 73.5% retained vs 26.5% churned. Accuracy alone is misleading.')

### Chart 2: Tenure Histogram by Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df[df['Churn']=='No']['tenure'].hist(bins=72, alpha=0.7, color='#1FA97A', label='No Churn', ax=ax)
df[df['Churn']=='Yes']['tenure'].hist(bins=72, alpha=0.7, color='#E5484D', label='Churned', ax=ax)
ax.set_title('Tenure Distribution by Churn', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()
print('💡 Insight: Churners cluster heavily in the first 1-12 months — early engagement is critical for retention.')

### Chart 3: MonthlyCharges Boxplot by Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
df.boxplot(column='MonthlyCharges', by='Churn', ax=ax,
           patch_artist=True,
           boxprops=dict(facecolor='#0E9F8A', alpha=0.5),
           medianprops=dict(color='#0F2040', linewidth=2))
ax.set_title('Monthly Charges by Churn Status', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_xlabel('Churn')
ax.set_ylabel('Monthly Charges ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()
print('💡 Insight: Churners pay significantly more per month (~$74 median vs ~$64 overall) — price sensitivity is a factor.')

### Chart 4: Churn Rate by Contract Type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean())
churn_by_contract.plot(kind='bar', color='#0E9F8A', edgecolor='white', ax=ax)
ax.set_title('Churn Rate by Contract Type', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: Month-to-month contracts have ~42% churn vs <5% for 2-year contracts — lock-in works.')

### Chart 5: Churn Rate by Internet Service

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
churn_by_internet = df.groupby('InternetService')['Churn'].apply(lambda x: (x=='Yes').mean())
churn_by_internet.plot(kind='bar', color='#0E9F8A', edgecolor='white', ax=ax)
ax.set_title('Churn Rate by Internet Service', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: Fiber optic users churn at ~42% — possibly due to higher prices or unmet expectations.')

### Chart 6: Churn Rate by Payment Method

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
churn_by_payment = df.groupby('PaymentMethod')['Churn'].apply(lambda x: (x=='Yes').mean())
churn_by_payment.plot(kind='bar', color='#0E9F8A', edgecolor='white', ax=ax)
ax.set_title('Churn Rate by Payment Method', fontsize=16, fontweight='bold', color='#0F2040')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: Electronic check users churn at ~45% — the least committed payment method.')

### Chart 7: Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr['TotalCharges'] = pd.to_numeric(df_corr['TotalCharges'], errors='coerce')
df_corr['Churn_num'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_corr[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn_r', center=0,
            linewidths=1, linecolor='white', ax=ax,
            xticklabels=['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'],
            yticklabels=['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'])
ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold', color='#0F2040')
plt.tight_layout()
plt.show()
print('💡 Insight: Tenure is negatively correlated with churn (-0.35) — longer customers are more loyal. TotalCharges correlates with tenure (0.83), not independently with churn.')

## Data Cleaning

In [ ]:
# Step 1: TotalCharges has 11 blank-space values, NOT NaN
print(f'Before cleaning shape: {df.shape}')
print(f'Blank TotalCharges (" "): {(df["TotalCharges"] == " ").sum()}')
print(f'NaN TotalCharges: {df["TotalCharges"].isna().sum()}')
print('\n⚠️ This is the trap: isna() returns 0 because blanks are " " not NaN!')

In [ ]:
# Fix: convert to numeric, coerce blanks to NaN, then drop
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'NaN TotalCharges after coerce: {df["TotalCharges"].isna().sum()}')
print(f'All 11 have tenure=0:')
print(df[df['TotalCharges'].isna()][['customerID', 'tenure', 'TotalCharges']])

In [ ]:
df = df.dropna(subset=['TotalCharges'])
print(f'Shape after drop: {df.shape}')  # Should be (7032, 21)

In [ ]:
df = df.drop(columns=['customerID'])
print(f'Shape after dropping customerID: {df.shape}')

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f'Churn distribution:')
print(df['Churn'].value_counts())
print(f'\nFinal cleaned shape: {df.shape}')